In [22]:
from dotenv import load_dotenv
from pathlib import Path
import sys
import os

# Walk up until we find the project root (folder with the .env)
current_path = Path().resolve()
for parent in [current_path] + list(current_path.parents):
    if (parent / ".env").exists():
        load_dotenv(parent / ".env")
        project_root = os.getenv("PROJECT_ROOT")
        print(project_root)
        sys.path.append(project_root)     
        break


%load_ext autoreload
%autoreload 2

C:\Users\Admin\Desktop\Emmanuel\beluga-call-pipeline\
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [23]:
import pandas as pd
import glob
import os

# Path to the snippets directory
snippets_dir = "../../data/evaluation_snippets/v2/"

# Get all folders in the snippets directory
snippet_folders = [f for f in os.listdir(snippets_dir) 
                   if os.path.isdir(os.path.join(snippets_dir, f))]

# Read all files from each folder's manual_verification directory
dfs = []
for folder in snippet_folders:
    manual_verification_dir = os.path.join(snippets_dir, folder, "manual_verification")
    
    # Check if manual_verification directory exists
    if not os.path.exists(manual_verification_dir):
        print(f"No manual_verification folder found in {folder}")
        continue
    
    # Get all files in the manual_verification directory
    manual_files = glob.glob(os.path.join(manual_verification_dir, "*"))
    
    # Read each file and add folder name as a column
    for file in manual_files:
        try:
            df = pd.read_csv(file, sep='\t', engine='python')
            df['site'] = folder  # Add folder name as a column
            context = df['context'].iloc[0]
            df["labeled_snippet_dir"] = os.path.join(snippets_dir, folder, context)

            # Extract annotator from filename: text between '.selections' and '.txt', remove leading character
            base = os.path.basename(file)
            if '.selections' in base and base.endswith('.txt'):
                annotator_raw = base.split('.selections')[1].replace('.txt', '')
                annotator = annotator_raw[1:] if len(annotator_raw) > 1 else ''
                df['annotator'] = annotator
                
            dfs.append(df)
            print(f"Read {os.path.basename(file)} from {folder}")
        except Exception as e:
            print(f"Could not read {file}: {e}")

# Concatenate all DataFrames into one
if dfs:
    all_manual_df = pd.concat(dfs, ignore_index=True)
    print(f"\nTotal rows: {len(all_manual_df)}")
    print(f"Datasets: {all_manual_df['site'].unique()}")
else:
    print("No data files found")
    all_manual_df = pd.DataFrame()

Read 201359382.170724094717.snippet.selections_VA.txt from BSM_2017
Read 201359382.170724103928.snippet.selections_VA.txt from BSM_2017
Read 201359382.170725060901.snippet.selections_VA.txt from BSM_2017
Read 201359382.210714135951.snippet.selections_VA.txt from CAC_2021
Read 201359382.210714142330.snippet.selections_VA.txt from CAC_2021
Read 201359382.210718085424.snippet.selections_VA.txt from CAC_2021
Read 201359382.210718085704.snippet.selections_VA.txt from CAC_2021
Read 201359382.210718182950.snippet.selections_JAA.txt from CAC_2021
Read 201359382.210718192740.snippet.selections_JAA.txt from CAC_2021
Read 201359382.210725161105.snippet.selections_JAA.txt from CAC_2021
Read 201359382.210803184644.snippet.selections_JAA.txt from CAC_2021
Read 201359382.210803191210.snippet.selections_JAA.txt from CAC_2021
Read 201359382.210804123124.snippet.selections-ma.txt from CAC_2021
Read 201359382.210804124936.snippet.selections-ma.txt from CAC_2021
Read 201359382.210804132905.snippet.selecti

In [24]:
labels_df = all_manual_df.copy()

labels_df = labels_df.drop(columns=["Selection", "View", "Channel", "Low Freq (Hz)", "High Freq (Hz)", "ECHO", "HFPC", "BBPC", "Whistle"])


In [25]:
labels_df["annotator"].value_counts(dropna=False)

annotator
JAA    1558
ma     1227
VA      836
Name: count, dtype: int64

In [26]:
labels_df.groupby("annotator")["GROUNDTRUTH"].value_counts(dropna=False)

annotator  GROUNDTRUTH
JAA        e              744
           a              439
           w              201
           ew             100
           eb              39
           eh              27
           b                6
           ae               1
           h                1
VA         NaN            783
           w               52
           b                1
ma         e              546
           a              284
           w              243
           ew              51
           ew?             29
           eb              21
           eh              16
           ewb             12
           h                9
           wb               5
           w?               2
           wh               2
           b                1
           e                1
           ew?b             1
           ew?h             1
           ewh              1
           w?b              1
           wbh              1
Name: count, dtype: int64

In [27]:
#TODO: Make sure this is still valide that they ddidnt' put a for absence, valeria in this case
labels_df["GROUNDTRUTH"].fillna("a", inplace=True)

C:\Users\Admin\AppData\Local\Temp\ipykernel_650424\1128083399.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  labels_df["GROUNDTRUTH"].fillna("a", inplace=True)


In [28]:
labels_df = labels_df.rename(columns={"snippet_filename": "labeled_snippet_filename"})

In [29]:
from pipeline.pipeline import get_hydrophone_model
from data_preprocessing.spectrogram.spectrogram_generator import HYDROPHONE_SENSITIVITY

# Apply get_hydrophone_model to the original_filename column for all rows
labels_df["HydrophoneModel"] = labels_df["original_filename"].apply(get_hydrophone_model)
labels_df["HydrophoneSensitivity"] = labels_df["HydrophoneModel"].apply(HYDROPHONE_SENSITIVITY.get_sensitivity)

In [30]:
labels_df["HydrophoneSensitivity"].value_counts()

HydrophoneSensitivity
-175.7    1829
-172.7    1792
Name: count, dtype: int64

In [31]:
labels_df.tail()

,Begin Time (s),End Time (s),GROUNDTRUTH,DETAILS,Notes,original_filename,labeled_snippet_filename,snippet_start_time,snippet_start_s,snippet_end_s,boat_labeling_file,boat_labeling_file_id,context,site,labeled_snippet_dir,annotator,HydrophoneModel,HydrophoneSensitivity
3616,164.0,165.0,a,NaN,NaN,5725.200801145954.wav,5725.200801152155.snippet.wav,2020-08-01 15:21:5,1321,1490,5725.200801145954.Table.1.selections.txt,2,Boat,KAM_2020,../../data/evaluation_snippets/v2/KAM_2020\Boat,ma,5725,-175.7
3617,165.0,166.0,a,NaN,NaN,5725.200801145954.wav,5725.200801152155.snippet.wav,2020-08-01 15:21:5,1321,1490,5725.200801145954.Table.1.selections.txt,2,Boat,KAM_2020,../../data/evaluation_snippets/v2/KAM_2020\Boat,ma,5725,-175.7
3618,166.0,167.0,a,NaN,NaN,5725.200801145954.wav,5725.200801152155.snippet.wav,2020-08-01 15:21:5,1321,1490,5725.200801145954.Table.1.selections.txt,2,Boat,KAM_2020,../../data/evaluation_snippets/v2/KAM_2020\Boat,ma,5725,-175.7
3619,167.0,168.0,a,NaN,NaN,5725.200801145954.wav,5725.200801152155.snippet.wav,2020-08-01 15:21:5,1321,1490,5725.200801145954.Table.1.selections.txt,2,Boat,KAM_2020,../../data/evaluation_snippets/v2/KAM_2020\Boat,ma,5725,-175.7
3620,168.0,169.0,a,NaN,NaN,5725.200801145954.wav,5725.200801152155.snippet.wav,2020-08-01 15:21:5,1321,1490,5725.200801145954.Table.1.selections.txt,2,Boat,KAM_2020,../../data/evaluation_snippets/v2/KAM_2020\Boat,ma,5725,-175.7


In [32]:
import pandas as pd

# Convert snippet_start_time to datetime and add the offset in seconds
labels_df["clip_start_time"] = pd.to_datetime(labels_df["snippet_start_time"]) + pd.to_timedelta(labels_df["Begin Time (s)"], unit='s')
labels_df["clip_end_time"] = pd.to_datetime(labels_df["snippet_start_time"]) + pd.to_timedelta(labels_df["End Time (s)"], unit='s')

In [33]:
labels_df.rename(columns={"site": "Site"}, inplace=True)
labels_df["Site"] = labels_df["Site"].str.split("_").str[0]

In [34]:
labels_df["clip_filename"] = labels_df["Site"] + "_" + labels_df["clip_start_time"].dt.strftime("%Y%m%d_%H%M%S%f").str[:-4] + ".wav"
# Check for duplicates in clip_filename
duplicate_clips = labels_df[labels_df.duplicated("clip_filename", keep=False)]
if not duplicate_clips.empty:
    print("Duplicates found in 'clip_filename':")
    display(duplicate_clips)
else:
    print("No duplicates found in 'clip_filename'.")

No duplicates found in 'clip_filename'.


In [35]:
def set_verif_flags(gt):
    if pd.isna(gt):
        return pd.Series([False, False, False, False])
    gt_str = str(gt)
    if 'a' in gt_str:
        return pd.Series([False, False, False, False])
    return pd.Series([
        'e' in gt_str,  # ECHO_verif
        'b' in gt_str,  # BBPC_verif
        'h' in gt_str,  # HFPC_verif
        'w' in gt_str   # Whislte_verif
    ])

labels_df[["ECHO", "BBPC", "HFPC", "Whistle"]] = labels_df["GROUNDTRUTH"].apply(set_verif_flags)
# Convert ECHO, BBPC, HFPC, Whistle columns to 0/1 integers
labels_df[["ECHO", "BBPC", "HFPC", "Whistle"]] = labels_df[["ECHO", "BBPC", "HFPC", "Whistle"]].astype(int)


In [37]:
labels_df["BBPC"].value_counts()

BBPC
0    3533
1      88
Name: count, dtype: int64

## Clipping to 1 second audio files

In [38]:
import os
import librosa
import soundfile as sf
from tqdm import tqdm

# Output directory
output_dir = "../../data/Verified_Dataset/clip_wavs"
os.makedirs(output_dir, exist_ok=True)

# Group by snippet to load each file only once
grouped = labels_df.groupby(["labeled_snippet_dir", "labeled_snippet_filename"])

for (snippet_dir, snippet_filename), group in tqdm(grouped, total=len(grouped)):
    # Build the full path to the source snippet
    source_path = os.path.join(snippet_dir, snippet_filename)
    
    try:
        # Load the entire snippet once
        y, sr = librosa.load(source_path, sr=None)
        
        # Extract all clips from this snippet
        for idx, row in group.iterrows():
            output_path = os.path.join(output_dir, row["clip_filename"])
            
            # Skip if already exists
            if os.path.exists(output_path):
                continue
            
            # Calculate sample indices
            start_sample = int(row["Begin Time (s)"] * sr)
            end_sample = int(row["End Time (s)"] * sr)
            
            # Extract and save the clip
            clip = y[start_sample:end_sample]
            sf.write(output_path, clip, sr)
            
    except Exception as e:
        print(f"Error processing {snippet_filename}: {e}")

100%|██████████| 28/28 [00:12<00:00,  2.25it/s]


In [41]:
labels_df["Boat"] = labels_df["context"].apply(lambda x: 1 if x == "Boat" else 0)
labels_df = labels_df.drop(columns=["context"])


In [42]:
labels_df["Boat"].value_counts()

Boat
1    2429
0    1192
Name: count, dtype: int64

In [44]:
labels_output_dir = "../../data/Verified_Dataset/labels"

labels_df.to_csv(os.path.join(labels_output_dir, "labels_eval_v2.csv"), index=False)